# Kafka issue types: BERT

In [ ]:
!pip install transformers datasets accelerate jira tqdm scikit-learn pandas numpy torch

import re
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score, confusion_matrix
from transformers import BertTokenizer, BertForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.auto import tqdm
from jira import JIRA

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

jira = JIRA('https://issues.apache.org/jira')
jql = 'project=KAFKA'

all_issues = []
start = 0
batch_size = 100

while True:
    issues = jira.search_issues(
        jql,
        startAt=start,
        maxResults=batch_size,
        fields=['summary', 'issuetype', 'status', 'created', 'reporter', 'description'])
    if not issues:
        break
    all_issues.extend(issues)
    print(f"Found {len(all_issues)} issues")
    start += batch_size
print(f"Found {len(all_issues)} issues total")

def extract_issue(issue):
    return {
        'bug': issue.key,
        'summary': issue.fields.summary or '',
        'description': issue.fields.description or '',
        'type': issue.fields.issuetype.name,
        'status': issue.fields.status.name,
        'created': issue.fields.created,
        'reporter': issue.fields.reporter.displayName if issue.fields.reporter else 'Unknown'
    }

data = [extract_issue(issue) for issue in all_issues]
df = pd.DataFrame(data)
df.to_csv('Kafka_issues.csv', index=False)

df['text'] = (df['summary'].fillna('') + ' ' + df['description'].fillna('')).str.strip()
df = df[df['text'].str.len() > 0].reset_index(drop=True)

MIN_CLASS_COUNT = 50
type_counts = df['type'].value_counts()
rare_types = type_counts[type_counts < MIN_CLASS_COUNT].index
if len(rare_types) > 0:
    print(f"Collapsing rare issue types into 'Other': {list(rare_types)}")
    df['type'] = df['type'].apply(lambda t: 'Other' if t in rare_types else t)

categories = sorted(df['type'].unique().tolist())
label_to_idx = {label: i for i, label in enumerate(categories)}
df['label'] = df['type'].map(label_to_idx)

print(f"\nModeling categories: {categories}")
print(df['type'].value_counts())

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

class BertJiraDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

train_dataset = BertJiraDataset(train_texts, train_labels, tokenizer)
test_dataset = BertJiraDataset(test_texts, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(categories)
)
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)
EPOCHS = 3
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for batch in progress:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        progress.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
macro_f1 = f1_score(all_labels, all_preds, average='macro')
weighted_f1 = f1_score(all_labels, all_preds, average='weighted')

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=categories))
print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

Using device: cuda
Found 19491 issues total

Modeling categories: ['Bug', 'Improvement', 'New Feature', 'Sub-task', 'Task', 'Test', 'Wish']
type
Bug            8895
Improvement    5012
Sub-task       3323
Task            871
Test            650
New Feature     606
Wish            134
Name: count, dtype: int64


Starting training...


Epoch 1/3 completed. Avg Loss: 1.0506


Epoch 2/3 completed. Avg Loss: 0.8273


Epoch 3/3 completed. Avg Loss: 0.6736
Training complete!
Evaluating on test set...


Accuracy: 0.6866
Macro F1: 0.4291
Weighted F1: 0.6757

Classification Report:
              precision    recall  f1-score   support

         Bug       0.83      0.82      0.82      1779
 Improvement       0.56      0.68      0.61      1003
 New Feature       0.39      0.24      0.30       121
    Sub-task       0.63      0.66      0.65       665
        Task       0.40      0.13      0.20       174
        Test       0.56      0.35      0.43       130
        Wish       0.00      0.00      0.00        27

    accuracy                           0.69      3899
   macro avg       0.48      0.41      0.43      3899
weighted avg       0.68      0.69      0.68      3899

Confusion Matrix:
[[1457  229    1   69    5   18    0]
 [ 152  683   33  116   13    6    0]
 [   4   71   29   17    0    0    0]
 [  65  130    7  440   13   10    0]
 [  35   75    2   37   23    2    0]
 [  41   22    1   17    4   45    0]
 [   5   20    1    1    0    0    0]]
